# Fast MRI from Undersampled k-Space · Notebook 2
### Learned reconstruction (U-Net) versus the classical methods

Notebook 1 reconstructed undersampled knee k-space with classical methods (zero-filled, per-coil FISTA, multi-coil FISTA) and evaluated them on four held-out test subjects. This notebook trains a **U-Net**, a convolutional neural network that learns to turn a zero-filled image into a clean one, and compares it with those methods **on exactly the same test slices, masks and metrics**.

Questions answered here:
1. Does a learned method beat classical compressed sensing at R = 4 and R = 8, and how consistently?
2. **Robustness:** how does a network trained at one acceleration behave at others, compared with FISTA, which needs no retraining?
3. **Training data size:** how does quality change with 3, 7 or 14 training subjects?
4. **Speed:** how long does each method take per slice?
5. **Where** do the methods fail (error maps)?
6. Does enforcing **data consistency** (putting the measured k-space lines back into the U-Net's output) help?
7. Are the differences **statistically significant**?

| Phase | Content |
|---|---|
| 0 | Setup, inputs and settings |
| 1 | Data: training, validation and test sets |
| 2 | The U-Net |
| 3 | Training |
| 4 | Evaluation on the test subjects |
| 5 | Comparison with the classical methods |
| 6 | Statistical tests |
| 7 | Summary |

**How to run:** Kaggle notebook with **Accelerator: GPU T4** (one GPU is used). Attach the output of the successful Notebook 1 version: *Add Input → Your Work → Notebook 1 → its output*. Internet is not needed. Expected run time is roughly 1–1.5 hours on a T4; Phase 3 prints its own estimate. Trained models and results are saved in `models/` and `results/`; re-running reloads them (set `FORCE_RETRAIN` / `FORCE_RERUN` to recompute).

## Phase 0 · Setup
Libraries, the location of Notebook 1's output, the compute device and all settings.

In [ ]:
import os, json, time, math, shutil
import numpy as np
import pandas as pd
import scipy
import scipy.fft as sfft
from scipy import ndimage
from functools import lru_cache
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity
from skimage.filters import threshold_otsu
from tqdm.auto import tqdm
from IPython.display import display

ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("work2")
INPUT_ROOT = "/kaggle/input"
RES_DIR, FIG_DIR, MODEL_DIR = (os.path.join(WORK, d) for d in ("results", "figures", "models"))
for d in (RES_DIR, FIG_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print(f"torch {torch.__version__} | numpy {np.__version__} | scipy {scipy.__version__}")
print("device:", torch.cuda.get_device_name(0) if AMP else "CPU (no GPU found: training will be very slow)")

pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "font.size": 10, "axes.titlesize": 10, "image.cmap": "gray"})

**Settings**

| Setting | Meaning |
|---|---|
| `R_TRAIN` | one U-Net is trained per acceleration factor |
| `DATA_SIZES` | extra R = 8 networks trained on fewer subjects (nested subsets of the training subjects) |
| `STEPS`, `BATCH`, `LR` | every network gets the same number of training steps, batch size and learning rate, so data-size results are comparable |
| `VAL_EVERY` | validation interval; the checkpoint with the best validation PSNR is kept |
| `CHANS`, `POOLS` | U-Net width (channels of the first level) and depth (number of down-sampling steps) |
| `REUSE_MODELS` | if `True`, trained networks found in an attached earlier output of *this* notebook are reused instead of retrained. Only use it when the attached Notebook 1 output is the same one those networks were trained on. |

In [ ]:
FORCE_RETRAIN = False
FORCE_RERUN = False
REUSE_MODELS = False

R_TRAIN = [4, 8]
DATA_SIZES = [3, 7]
STEPS, BATCH, LR = 4000, 8, 1e-3
VAL_EVERY = 200
CHANS, POOLS = 32, 4
TRAIN_SEED = 1234
ESPIRIT = {"kw": 6, "thresh": 0.02, "crop": 0.0}

In [ ]:
def find_nb1_output():
    for dirpath, _, files in os.walk(INPUT_ROOT):
        if "project_config.json" in files and os.path.basename(dirpath) == "results":
            return os.path.dirname(dirpath)
    raise FileNotFoundError("Notebook 1 output not found: attach it with Add Input (results/project_config.json is required)")

NB1 = find_nb1_output()
CFG = json.load(open(os.path.join(NB1, "results", "project_config.json")))
N_ACS, SEEDS, R_CURVE = CFG["n_acs"], CFG["seeds"], CFG["r_curve"]
DEV, TEST, TRAIN = CFG["development_subject"], CFG["test_subjects"], CFG["train_subjects"]
frozen = pd.read_csv(os.path.join(NB1, "results", "frozen_settings.csv"))
print("Notebook 1 output:", NB1)
print(f"development subject {DEV} | test subjects {TEST} | training subjects {TRAIN}")
display(frozen)

### Functions shared with Notebook 1
Copied unchanged from Notebook 1 so that masks, normalisation, metrics and the classical reconstructions are identical: FFT helpers, root-sum-of-squares, the mask generator, the foreground mask and metrics, and (for the speed test and the visual comparison) the wavelet/FISTA code and ESPIRiT coil maps.

New here is `data_consistency`: the U-Net's magnitude image gets the phase of a low-resolution image from the fully sampled centre, is projected through each coil's ESPIRiT map into k-space, the **measured lines are put back exactly**, and the result is transformed back and combined by RSS. The output can then no longer contradict the measured data on the lines that were acquired.

In [ ]:
def fft2c(x):
    return sfft.fftshift(sfft.fft2(sfft.ifftshift(x, axes=(-2, -1)), norm="ortho", workers=-1), axes=(-2, -1))

def ifft2c(x):
    return sfft.fftshift(sfft.ifft2(sfft.ifftshift(x, axes=(-2, -1)), norm="ortho", workers=-1), axes=(-2, -1))

def rss(coil_imgs):
    return np.sqrt((np.abs(coil_imgs) ** 2).sum(axis=0))

def make_mask(n, R, pattern="vd", seed=0, n_acs=N_ACS):
    c = n // 2
    mask = np.zeros(n, bool)
    mask[c - n_acs // 2: c - n_acs // 2 + n_acs] = True
    outside = np.flatnonzero(~mask)
    n_extra = max(int(round(n / R)) - int(mask.sum()), 0)
    rng = np.random.default_rng(seed)
    if pattern == "equispaced":
        pick = outside[np.round(np.linspace(0, len(outside) - 1, n_extra)).astype(int)]
    elif pattern == "random":
        pick = rng.choice(outside, n_extra, replace=False)
    elif pattern == "vd":
        p = (1 - np.abs(outside - c) / c) ** 3
        pick = rng.choice(outside, n_extra, replace=False, p=p / p.sum())
    else:
        raise ValueError(f"unknown pattern {pattern}")
    mask[pick] = True
    return mask

def undersample(ks, mask):
    return ks * mask[None, :, None]

def foreground_mask(ref):
    sm = ndimage.gaussian_filter(ref, 2)
    m = sm > 0.5 * threshold_otsu(sm)
    m = ndimage.binary_closing(m, iterations=3)
    m = ndimage.binary_fill_holes(m)
    lab, n = ndimage.label(m)
    if n > 1:
        m = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    return m

def metrics(img, ref, fg):
    img, ref = np.asarray(img, np.float64), np.asarray(ref, np.float64)
    err = img - ref
    ssim, smap = structural_similarity(ref, img, data_range=1.0, full=True)
    return {"psnr": float(10 * np.log10(1.0 / np.mean(err ** 2))), "ssim": float(ssim),
            "ssim_fg": float(smap[fg].mean()), "nmse": float(np.sum(err ** 2) / np.sum(ref ** 2))}

WAVELET, LEVEL = "db4", 4

@lru_cache(maxsize=None)
def approx_shape(ny, nx):
    return pywt.wavedec2(np.zeros((ny, nx), np.float32), WAVELET, mode="periodization", level=LEVEL)[0].shape

def wt(x):
    ri = np.stack([x.real, x.imag])
    arr, sl = pywt.coeffs_to_array(pywt.wavedec2(ri, WAVELET, mode="periodization", level=LEVEL, axes=(-2, -1)), axes=(-2, -1))
    return arr[0] + 1j * arr[1], sl

def iwt(c, sl):
    ri = np.stack([c.real, c.imag])
    out = pywt.waverec2(pywt.array_to_coeffs(ri, sl, output_format="wavedec2"), WAVELET, mode="periodization", axes=(-2, -1))
    return out[0] + 1j * out[1]

def prox(x, lam):
    c, sl = wt(x)
    a, b = approx_shape(*x.shape[-2:])
    coarse = c[..., :a, :b].copy()
    c = c * np.maximum(1 - lam / np.maximum(np.abs(c), 1e-12), 0)
    c[..., :a, :b] = coarse
    return iwt(c, sl).astype(x.dtype)

def fista(A, AH, y, lam, n_iter, out_fn):
    x = AH(y)
    z, t = x.copy(), 1.0
    for _ in range(n_iter):
        xn = prox(z - AH(A(z) - y), lam)
        tn = (1 + math.sqrt(1 + 4 * t * t)) / 2
        z = xn + ((t - 1) / tn) * (xn - x)
        x, t = xn, tn
    return out_fn(x)

def recon_zf(y):
    return rss(ifft2c(y))

def recon_pc(y, mask, lam, n_iter):
    M = mask[None, :, None]
    return fista(lambda x: M * fft2c(x), lambda r: ifft2c(M * r), y, lam, n_iter, rss)

def recon_mc(y, mask, maps, lam, n_iter):
    M = mask[None, :, None]
    return fista(lambda x: M * fft2c(maps * x[None]), lambda r: (maps.conj() * ifft2c(M * r)).sum(axis=0),
                 y, lam, n_iter, np.abs)

def espirit_maps(ks, n_cal=N_ACS, kw=6, thresh=0.02, crop=0.0):
    nc, ny, nx = ks.shape
    cal = ks[:, ny // 2 - n_cal // 2: ny // 2 - n_cal // 2 + n_cal, nx // 2 - n_cal // 2: nx // 2 - n_cal // 2 + n_cal]
    blocks = np.lib.stride_tricks.sliding_window_view(cal.astype(np.complex128), (kw, kw), axis=(1, 2))
    A = blocks.reshape(nc, -1, kw * kw).transpose(1, 0, 2).reshape(-1, nc * kw * kw)
    _, s, vh = np.linalg.svd(A, full_matrices=False)
    kernels = vh[s > thresh * s.max()].reshape(-1, nc, kw, kw)
    AHA = np.zeros((ny * nx, nc, nc), np.complex64)
    y0, x0 = ny // 2 - kw // 2, nx // 2 - kw // 2
    for i in range(0, len(kernels), 32):
        ker = kernels[i:i + 32]
        pad = np.zeros((len(ker), nc, ny, nx), np.complex64)
        pad[:, :, y0:y0 + kw, x0:x0 + kw] = ker
        img = np.ascontiguousarray(ifft2c(pad).reshape(len(ker), nc, -1).transpose(2, 1, 0))
        AHA += img @ img.conj().transpose(0, 2, 1)
    AHA *= ny * nx / kw ** 2
    w, v = np.linalg.eigh(AHA)
    maps = v[:, :, -1].T.reshape(nc, ny, nx)
    eig = w[:, -1].reshape(ny, nx)
    maps = maps * np.exp(-1j * np.angle(maps.sum(axis=0)))[None]
    return (maps * (eig > crop)[None]).astype(np.complex64)

def acs_mask(n):
    m = np.zeros(n, bool)
    m[n // 2 - N_ACS // 2: n // 2 - N_ACS // 2 + N_ACS] = True
    return m

def data_consistency(img, y, mask, maps):
    low = (maps.conj() * ifft2c(undersample(y, acs_mask(y.shape[1])))).sum(axis=0)
    x = img * np.exp(1j * np.angle(low))
    k_dc = np.where(mask[None, :, None], y, fft2c(maps * x[None]))
    return rss(ifft2c(k_dc)).astype(np.float32)

def frozen_setting(name):
    rows = frozen[frozen.method == name]
    if rows.empty and name.startswith("multi-coil (R="):
        rows = frozen[frozen.method == "multi-coil (all R)"]
    if rows.empty:
        raise KeyError(f"setting '{name}' not found in Notebook 1's frozen_settings.csv")
    return float(rows.iloc[0]["lambda"]), int(rows.iloc[0]["iterations"])

def show(ax, img, title=None, vmin=0, vmax=1, cmap="gray"):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        ax.set_title(title)
    ax.axis("off")

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name))
    plt.show()

def fmt_ms(s, digits):
    return f"{s.mean():.{digits}f} ± {s.std():.{digits}f}"

def cached(name, fn):
    path = os.path.join(RES_DIR, name)
    if os.path.exists(path) and not FORCE_RERUN:
        return json.load(open(path))
    out = fn()
    with open(path, "w") as f:
        json.dump(out, f, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    return out

## Phase 1 · Data
All data comes from Notebook 1's output:
* **training:** `unet_data/` of the training subjects (40 slices each, raw coil k-space);
* **validation:** `unet_data/` of the development subject. It is used only to pick the best checkpoint, just as the same subject was used to tune FISTA;
* **test:** `data/` of the four test subjects: the very slices Notebook 1 evaluated.

Every slice is normalised exactly as in Notebook 1 (k-space divided by the maximum of its fully sampled RSS image, so the ground truth peaks at 1). Training and validation k-space is stored on the GPU; each training batch gets **new random masks**, so the network never sees the same undersampled input twice. Training and validation masks use seeds from 10 000 upwards and never reuse the test seeds.

In [ ]:
def load_npz(folder, s):
    d = np.load(os.path.join(NB1, folder, f"subject{s:02d}.npz"))
    return d["kspace"], d["slices"]

def norm_slice(ks):
    ref = rss(ifft2c(ks))
    scale = ref.max()
    return (ks / scale).astype(np.complex64), (ref / scale).astype(np.float32)

def build_set(subjects):
    counts = [len(np.load(os.path.join(NB1, "unet_data", f"subject{s:02d}.npz"))["slices"]) for s in subjects]
    k0, _ = load_npz("unet_data", subjects[0])
    nc, ny, nx = k0.shape[1:]
    k_t = torch.empty((sum(counts), nc, ny, nx), dtype=torch.complex64, device=DEVICE)
    ref_t = torch.empty((sum(counts), 1, ny, nx), dtype=torch.float32, device=DEVICE)
    owner, i = np.zeros(sum(counts), int), 0
    for s in tqdm(subjects, desc="loading"):
        k_all, _ = load_npz("unet_data", s)
        for ks in k_all:
            kn, ref = norm_slice(ks)
            k_t[i], ref_t[i, 0], owner[i] = torch.from_numpy(kn), torch.from_numpy(ref), s
            i += 1
        del k_all
    return k_t, ref_t, owner

train_k, train_t, train_owner = build_set(TRAIN)
val_k, val_t, _ = build_set([DEV])
NC, NY, NX = train_k.shape[1:]
print(f"training: {len(train_k)} slices from {len(TRAIN)} subjects | validation: {len(val_k)} slices | "
      f"{NC} coils, {NY} x {NX} | GPU memory used: {torch.cuda.memory_allocated() / 1e9 if AMP else 0:.2f} GB")

def zero_filled_t(k, masks):
    img = torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k * masks[:, None, :, None], dim=(-2, -1)),
                                             norm="ortho"), dim=(-2, -1))
    return torch.sqrt((img.abs() ** 2).sum(dim=1, keepdim=True))

def masks_t(R, seeds):
    return torch.from_numpy(np.stack([make_mask(NY, R, "vd", int(s)) for s in seeds])).to(DEVICE)

val_masks = {R: masks_t(R, 10_000 + np.arange(len(val_k))) for R in R_TRAIN}

rng_show = np.random.default_rng(0)
pick = rng_show.choice(len(train_k), 3, replace=False)
x_show = zero_filled_t(train_k[pick], masks_t(8, 20_000 + pick)).cpu().numpy()[:, 0]
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for j, i in enumerate(pick):
    show(ax[0, j], train_t[i, 0].cpu().numpy(), f"target: subject {train_owner[i]}")
    show(ax[1, j], x_show[j], "network input: zero-filled, R = 8")
plt.tight_layout()
save(fig, "nb2_fig00_training_pairs.png")

**Training pairs:** the network sees the bottom row (zero-filled, blurred and aliased) and is trained to produce the top row (the fully sampled image).

## Phase 2 · The U-Net
A standard U-Net in the configuration of the fastMRI baseline:
* **Encoder:** `POOLS` levels, each two 3×3 convolutions (instance normalisation, LeakyReLU) followed by 2×2 average pooling. The image shrinks 320 → 160 → 80 → 40 → 20 while the number of channels doubles, so the deepest level sees the whole knee at once.
* **Decoder:** transposed convolutions enlarge the image back step by step; at each level the encoder's features of the same size are concatenated (**skip connections**), which carries fine detail across.
* **Residual output:** the network predicts a *correction* that is added to the zero-filled input, so it only has to learn what is missing. The last layer starts at zero, so training begins from the zero-filled image itself.
* **Loss:** L1 (mean absolute error) between output and fully sampled image; Adam optimiser with a cosine learning-rate decay; mixed precision on the GPU.

In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm2d(cout), nn.LeakyReLU(0.2, inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm2d(cout), nn.LeakyReLU(0.2, inplace=True))

class UNet(nn.Module):
    def __init__(self, chans=32, pools=4):
        super().__init__()
        self.down = nn.ModuleList([ConvBlock(1, chans)])
        ch = chans
        for _ in range(pools - 1):
            self.down.append(ConvBlock(ch, ch * 2))
            ch *= 2
        self.bottom = ConvBlock(ch, ch * 2)
        self.up_t, self.up_c = nn.ModuleList(), nn.ModuleList()
        for _ in range(pools):
            self.up_t.append(nn.ConvTranspose2d(ch * 2, ch, 2, stride=2))
            self.up_c.append(ConvBlock(ch * 2, ch))
            ch //= 2
        self.out = nn.Conv2d(chans, 1, 1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        skips, h = [], x
        for block in self.down:
            h = block(h)
            skips.append(h)
            h = F.avg_pool2d(h, 2)
        h = self.bottom(h)
        for up, block in zip(self.up_t, self.up_c):
            h = block(torch.cat([up(h), skips.pop()], dim=1))
        return x + self.out(h)

n_params = sum(p.numel() for p in UNet(CHANS, POOLS).parameters())
print(f"U-Net: {POOLS} levels, {CHANS} base channels, {n_params / 1e6:.2f} M parameters")

## Phase 3 · Training
Networks trained (all with the same `STEPS`, batch size and learning rate):

| Network | Acceleration | Training subjects |
|---|---|---|
| `unet_R4` | 4 | all training subjects |
| `unet_R8` | 8 | all training subjects |
| `unet_R8_3subj`, `unet_R8_7subj` | 8 | nested random subsets (3 ⊂ 7 ⊂ all) |

Every `VAL_EVERY` steps the network is scored on the development subject (fixed validation masks); the checkpoint with the highest validation PSNR is kept. The test subjects are never used.

In [ ]:
def predict(model, k, masks, batch=8):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(k), batch):
            x = zero_filled_t(k[i:i + batch], masks[i:i + batch])
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                outs.append(model(x).float())
    model.train()
    return torch.cat(outs)

def psnr_t(a, b):
    return 10 * torch.log10(1.0 / ((a - b) ** 2).mean(dim=(1, 2, 3)))

def train(name, R, subjects, steps=STEPS):
    path, hist_path = os.path.join(MODEL_DIR, f"{name}.pt"), os.path.join(MODEL_DIR, f"{name}_history.json")
    model = UNet(CHANS, POOLS).to(DEVICE)
    if os.path.exists(path) and os.path.exists(hist_path) and not FORCE_RETRAIN:
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        return model.eval(), json.load(open(hist_path))
    pool = np.flatnonzero(np.isin(train_owner, subjects))
    rng = np.random.default_rng(TRAIN_SEED)
    torch.manual_seed(TRAIN_SEED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    history, best, running, t0 = [], -np.inf, [], time.time()
    for step in tqdm(range(1, steps + 1), desc=name, mininterval=30):
        b = torch.as_tensor(rng.choice(pool, BATCH), device=DEVICE)
        with torch.no_grad():
            x = zero_filled_t(train_k[b], masks_t(R, rng.integers(10_000, 2**31 - 1, BATCH)))
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(x)
        loss = F.l1_loss(out.float(), train_t[b])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        running.append(loss.item())
        if step % VAL_EVERY == 0 or step == steps:
            val = psnr_t(predict(model, val_k, val_masks[R]), val_t).mean().item()
            history.append({"step": step, "train_l1": float(np.mean(running)), "val_psnr": val,
                            "seconds": time.time() - t0})
            running = []
            if val > best:
                best = val
                torch.save(model.state_dict(), path + ".part")
                os.replace(path + ".part", path)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    with open(hist_path, "w") as f:
        json.dump(history, f)
    return model.eval(), history

subset_order = [int(s) for s in np.random.default_rng(TRAIN_SEED).permutation(TRAIN)]
PLAN = {f"unet_R{R}": (R, TRAIN) for R in R_TRAIN}
PLAN.update({f"unet_R8_{n}subj": (8, sorted(subset_order[:n])) for n in DATA_SIZES if n < len(TRAIN)})
LABEL = {f"unet_R{R}": f"U-Net trained at R={R}" for R in R_TRAIN}
LABEL.update({f"unet_R8_{n}subj": f"U-Net R=8, {n} subjects" for n in DATA_SIZES})
display(pd.DataFrame([{"network": k, "R": v[0], "training subjects": v[1]} for k, v in PLAN.items()]))

probe = UNet(CHANS, POOLS).to(DEVICE)
opt_p = torch.optim.Adam(probe.parameters(), lr=LR)
rng_p = np.random.default_rng(0)
for i in range(13):
    if i == 3:
        if AMP:
            torch.cuda.synchronize()
        t0 = time.time()
    b = torch.as_tensor(rng_p.choice(len(train_k), BATCH), device=DEVICE)
    x = zero_filled_t(train_k[b], masks_t(8, rng_p.integers(10_000, 2**31 - 1, BATCH)))
    loss = F.l1_loss(probe(x), train_t[b])
    opt_p.zero_grad(); loss.backward(); opt_p.step()
if AMP:
    torch.cuda.synchronize()
per_step = (time.time() - t0) / 10
del probe, opt_p
print(f"about {per_step * 1000:.0f} ms per step (full precision probe) -> roughly "
      f"{per_step * STEPS * len(PLAN) / 60:.0f} min for {len(PLAN)} networks (mixed precision is usually faster)")

In [ ]:
if REUSE_MODELS:
    reused = 0
    for dirpath, _, files in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) != "models":
            continue
        for f in files:
            if f.split(".")[0].replace("_history", "") in PLAN and not os.path.exists(os.path.join(MODEL_DIR, f)):
                shutil.copy(os.path.join(dirpath, f), os.path.join(MODEL_DIR, f))
                reused += 1
    print(f"files of trained networks reused from attached inputs: {reused}")

MODELS, HISTORY = {}, {}
for name, (R, subjects) in PLAN.items():
    t0 = time.time()
    MODELS[name], HISTORY[name] = train(name, R, subjects)
    best = max(HISTORY[name], key=lambda h: h["val_psnr"])
    print(f"{name}: best validation PSNR {best['val_psnr']:.2f} dB at step {best['step']} ({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for name, hist in HISTORY.items():
    steps_ = [h["step"] for h in hist]
    ax[0].plot(steps_, [h["train_l1"] for h in hist], label=LABEL[name])
    ax[1].plot(steps_, [h["val_psnr"] for h in hist], label=LABEL[name])
ax[0].set_title("training loss (L1)"); ax[0].set_yscale("log")
ax[1].set_title(f"validation PSNR (subject {DEV})")
for a in ax:
    a.set_xlabel("training step"); a.legend(fontsize=8)
ax[1].set_ylabel("dB")
plt.tight_layout()
save(fig, "nb2_fig01_training_curves.png")

**Reading the training curves:** the training loss should fall steadily. Validation PSNR should rise and level off; if it starts to fall while the training loss keeps dropping, the network is overfitting (memorising the training knees), and the saved checkpoint is the one from before that point. Networks trained on fewer subjects are expected to level off lower or overfit earlier.

## Phase 4 · Evaluation on the test subjects
Every test slice of Notebook 1 is reconstructed by every network, with the **same variable-density masks (seeds `SEEDS`) at every R in `R_CURVE`** as in Notebook 1. The R = 4 and R = 8 networks are run at all R (robustness); the smaller-data networks only at R = 8.

**Data consistency:** for the network trained at the matching R, the output is also post-processed with `data_consistency` (coil maps estimated from the ACS block, as in Notebook 1); no retraining is involved.

**Check:** the zero-filled reconstruction is recomputed here and compared with Notebook 1's stored values. They must agree, which confirms that masks, normalisation and metrics are identical in both notebooks.

In [ ]:
def run_eval():
    rows = []
    for s in TEST:
        k_all, slices = load_npz("data", s)
        for pos in tqdm(range(len(k_all)), desc=f"subject {s}"):
            ks, ref = norm_slice(k_all[pos])
            fg = foreground_mask(ref)
            ny = ks.shape[1]
            k_rep = torch.from_numpy(ks).to(DEVICE).unsqueeze(0).repeat(len(SEEDS), 1, 1, 1)
            maps = espirit_maps(ks, **ESPIRIT)
            base = {"subject": s, "pos": pos, "slice": int(slices[pos]), "pattern": "vd"}
            for R in R_CURVE:
                masks = [make_mask(ny, R, "vd", seed) for seed in SEEDS]
                for seed, m in zip(SEEDS, masks):
                    rows.append({**base, "R": R, "seed": seed, "method": "zero-filled (recomputed)",
                                 **metrics(recon_zf(undersample(ks, m)), ref, fg)})
                m_t = torch.from_numpy(np.stack(masks)).to(DEVICE)
                for name, model in MODELS.items():
                    if name not in [f"unet_R{r}" for r in R_TRAIN] and R != PLAN[name][0]:
                        continue
                    out = predict(model, k_rep, m_t).cpu().numpy()[:, 0]
                    for seed, img in zip(SEEDS, out):
                        rows.append({**base, "R": R, "seed": seed, "method": LABEL[name], **metrics(img, ref, fg)})
                    if name == f"unet_R{R}":
                        for seed, m, img in zip(SEEDS, masks, out):
                            dc = data_consistency(img, undersample(ks, m), m, maps)
                            rows.append({**base, "R": R, "seed": seed, "method": "U-Net + data consistency", **metrics(dc, ref, fg)})
    return rows

t0 = time.time()
nb2 = pd.DataFrame(cached("nb2_unet_eval.json", run_eval))
nb1 = pd.concat([pd.DataFrame(json.load(open(os.path.join(NB1, "results", f"eval_subject{s:02d}.json")))) for s in TEST])
print(f"U-Net evaluation: {len(nb2)} rows ({time.time() - t0:.0f} s) | Notebook 1 results: {len(nb1)} rows")

KEYS = ["subject", "pos", "R", "seed", "pattern"]
chk = nb2[nb2.method == "zero-filled (recomputed)"].merge(nb1[nb1.method == "zero-filled"], on=KEYS, suffixes=("_nb2", "_nb1"))
print(f"check against Notebook 1 ({len(chk)} matched cases): max |PSNR difference| = "
      f"{(chk.psnr_nb2 - chk.psnr_nb1).abs().max():.2e} dB, max |SSIM difference| = {(chk.ssim_nb2 - chk.ssim_nb1).abs().max():.2e}")

unet_main = nb2[nb2.method.isin([LABEL[f"unet_R{R}"] for R in R_TRAIN])].copy()
unet_main = unet_main[unet_main.apply(lambda r: r.method == LABEL[f"unet_R{r.R}"] if r.R in R_TRAIN else False, axis=1)]
unet_main["method"] = "U-Net"
ALL = pd.concat([nb1[nb1.pattern == "vd"], nb2[nb2.method != "zero-filled (recomputed)"], unet_main], ignore_index=True)

## Phase 5 · Comparison with the classical methods

### 5a · Main comparison at the trained accelerations
For each R, `U-Net` is the network trained at that R. *cases* = test slices × mask seeds. Δ columns are paired differences on the same slice and mask.

In [ ]:
METRICS = {"psnr": 2, "ssim": 4, "ssim_fg": 4, "nmse": 4}

def paired(df, method, base):
    a = df[df.method == method].set_index(KEYS)[list(METRICS)]
    b = df[df.method == base].set_index(KEYS)[list(METRICS)]
    a, b = a.align(b, join="inner")
    return a - b

NAMES = {"psnr": "PSNR (dB)", "ssim": "SSIM", "ssim_fg": "SSIM (knee)", "nmse": "NMSE"}
HIGHER = {"psnr": True, "ssim": True, "ssim_fg": True, "nmse": False}

def means_table(df, methods, bases):
    rows = []
    for R in sorted(df.R.unique()):
        dR = df[df.R == R]
        for m in methods:
            s = dR[dR.method == m]
            row = {"R": R, "method": m}
            row.update({NAMES[k]: round(s[k].mean(), d) for k, d in METRICS.items()})
            for b in bases:
                if m != b and m in dR.method.values and b in dR.method.values:
                    d = paired(dR, m, b)
                    row[f"PSNR better than {b}"] = f"{(d.psnr > 0).sum()} of {len(d)}"
            rows.append(row)
    return pd.DataFrame(rows).fillna("")

def bar_chart(df, methods, fname, title):
    Rs = sorted(df.R.unique())
    x, w = np.arange(len(Rs)), 0.8 / len(methods)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.4))
    for a, k in zip(ax, ["psnr", "ssim", "ssim_fg", "nmse"]):
        for i, m in enumerate(methods):
            vals = [df[(df.R == R) & (df.method == m)][k].mean() for R in Rs]
            bars = a.bar(x + (i - (len(methods) - 1) / 2) * w, vals, w, label=m)
            a.bar_label(bars, fmt="%.1f" if k == "psnr" else "%.3f", fontsize=7)
        a.set_xticks(x); a.set_xticklabels([f"R = {R}" for R in Rs])
        a.set_title(f"{NAMES[k]} ({'higher' if HIGHER[k] else 'lower'} is better)")
        if k == "psnr":
            a.set_ylim(df[df.method.isin(methods)].groupby(["R", "method"]).psnr.mean().min() - 3, None)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

def box_plots(df, methods, fname, title):
    Rs = sorted(df.R.unique())
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    for a, k in zip(ax, ["psnr", "ssim_fg", "nmse"]):
        data = [df[(df.R == R) & (df.method == m)][k].values for R in Rs for m in methods]
        a.boxplot(data)
        a.set_xticks(range(1, len(data) + 1))
        a.set_xticklabels([f"{m}\nR={R}" for R in Rs for m in methods], fontsize=7)
        a.set_title(f"{NAMES[k]} per test case ({'higher' if HIGHER[k] else 'lower'} is better)")
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

def subject_chart(df, methods, fname, title):
    Rs, subs = sorted(df.R.unique()), sorted(df.subject.unique())
    x, w = np.arange(len(subs)), 0.8 / len(methods)
    fig, ax = plt.subplots(1, len(Rs), figsize=(7.5 * len(Rs), 4.4), squeeze=False)
    for a, R in zip(ax[0], Rs):
        for i, m in enumerate(methods):
            vals = [df[(df.R == R) & (df.method == m) & (df.subject == s)].psnr.mean() for s in subs]
            a.bar(x + (i - (len(methods) - 1) / 2) * w, vals, w, label=m)
        a.set_xticks(x); a.set_xticklabels([f"subject {s}" for s in subs])
        a.set_ylim(df[(df.R == R) & df.method.isin(methods)].groupby(["subject", "method"]).psnr.mean().min() - 2, None)
        a.set_title(f"mean PSNR (dB) per test subject, R = {R}")
    ax[0, 0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

METHODS = ["zero-filled", "per-coil", "multi-coil", "U-Net", "U-Net + data consistency"]
main = ALL[ALL.R.isin(R_TRAIN) & ALL.method.isin(METHODS)]
tableA = means_table(main, METHODS, ["zero-filled", "multi-coil"])
tableA.to_csv(os.path.join(RES_DIR, "nb2_tableA_main.csv"), index=False)
print("Mean scores on the test subjects (same slices and masks for every method)")
display(tableA)
bar_chart(main, METHODS, "nb2_fig02a_all_methods_bars.png", "All methods on the test subjects")
box_plots(main, ["zero-filled", "multi-coil", "U-Net", "U-Net + data consistency"], "nb2_fig02b_all_methods_boxplots.png",
          "Spread over test slices and masks")
subject_chart(main, METHODS, "nb2_fig02c_all_methods_per_subject.png", "The same comparison for each test subject")
for R in R_TRAIN:
    d = paired(main[main.R == R], "U-Net", "multi-coil")
    print(f"R = {R}: U-Net vs multi-coil FISTA: {d.psnr.mean():+.2f} dB PSNR, {d.ssim_fg.mean():+.3f} SSIM (knee), "
          f"better on {(d.psnr > 0).sum()} of {len(d)} cases")

**Reading the charts:** bars are means over all test cases; box plots show the spread; the per-subject chart shows whether the ranking holds for every test person. The table's last columns count the cases (same slice, same mask) where a method beats zero-filled and multi-coil FISTA. Both families had one development subject for their choices (FISTA's λ and iterations, the U-Net's checkpoint); the U-Net additionally learned from the training subjects.

### 5b · Robustness: quality across accelerations
FISTA was run with one frozen setting at every R. Each U-Net is applied at every R, including accelerations it was never trained for.

In [ ]:
curve_methods = ["zero-filled", "multi-coil", LABEL["unet_R4"], LABEL["unet_R8"]]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for j, (metric, ylabel) in enumerate([("psnr", "PSNR (dB)"), ("ssim_fg", "SSIM inside the knee")]):
    for m, style in zip(curve_methods, ["o--", "o-", "s-", "^-"]):
        g = ALL[ALL.method == m].groupby("R")[metric]
        ax[j].errorbar(g.mean().index, g.mean().values, yerr=g.std().values, fmt=style, capsize=3, label=m)
    g = ALL[ALL.method == "per-coil"].groupby("R")[metric].mean()
    ax[j].plot(g.index, g.values, "D", color="gray", label="per-coil")
    ax[j].set_xlabel("acceleration factor R"); ax[j].set_ylabel(ylabel); ax[j].set_xticks(R_CURVE); ax[j].legend(fontsize=8)
ax[0].set_title("Quality vs acceleration on the test subjects (mean ± sd)")
plt.tight_layout()
save(fig, "nb2_fig02_robustness.png")

tableC = ALL[ALL.method.isin(curve_methods + ["per-coil"])].groupby(["R", "method"])["psnr"].mean().unstack("method")
tableC = tableC[[c for c in ["zero-filled", "per-coil", "multi-coil", LABEL["unet_R4"], LABEL["unet_R8"]] if c in tableC]]
tableC.to_csv(os.path.join(RES_DIR, "nb2_tableC_robustness_psnr.csv"))
print("Mean PSNR (dB) by acceleration")
display(tableC)

**Reading Figure 2 / Table C:** at its own R each network should be at its best. Away from it, a network may degrade: one trained at R = 8 expects heavy blur and can over-correct a mildly undersampled image; one trained at R = 4 may not know how to fill the larger gaps at R = 8. A method that needs no retraining (FISTA) does not have this problem, but it showed its own weakness at R = 2 in Notebook 1.

### 5c · How much training data is needed?
R = 8 networks trained on nested subsets of the training subjects, all with the same number of training steps, evaluated on the same test cases.

In [ ]:
size_rows = []
for name, (R, subjects) in PLAN.items():
    if R != 8:
        continue
    d = ALL[(ALL.method == LABEL[name]) & (ALL.R == 8)]
    size_rows.append({"network": LABEL[name], "training subjects": len(subjects), "training slices": int(np.isin(train_owner, subjects).sum()),
                      "best validation PSNR": max(h["val_psnr"] for h in HISTORY[name]),
                      "test PSNR": d.psnr.mean(), "test SSIM (knee)": d.ssim_fg.mean()})
tableD = pd.DataFrame(size_rows).sort_values("training subjects")
tableD.to_csv(os.path.join(RES_DIR, "nb2_tableD_data_size.csv"), index=False)
display(tableD)

ref8 = ALL[ALL.R == 8].groupby("method")["psnr"].mean()
fig, ax = plt.subplots(figsize=(6.5, 4.3))
ax.plot(tableD["training subjects"], tableD["test PSNR"], "o-", label="U-Net (R = 8)")
ax.axhline(ref8["multi-coil"], color="C1", ls="--", label="multi-coil FISTA")
ax.axhline(ref8["zero-filled"], color="gray", ls=":", label="zero-filled")
ax.set_xlabel("training subjects"); ax.set_ylabel("test PSNR (dB)"); ax.set_xticks(tableD["training subjects"])
ax.set_title("U-Net quality vs amount of training data (R = 8)"); ax.legend()
plt.tight_layout()
save(fig, "nb2_fig03_data_size.png")

**Reading Figure 3:** if the curve is still rising at the full training set, more subjects would probably help, i.e. the dataset size is limiting the U-Net. A flat curve means the network has learned what this data can teach it.

### 5d · Speed
Time per slice on one test slice at R = 8 (after a warm-up run). FISTA runs on the CPU with Notebook 1's frozen settings; the multi-coil time includes estimating the ESPIRiT coil maps. Training time of the U-Net is not included (it is paid once).

In [ ]:
def timed(fn, reps):
    fn()
    if AMP:
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        fn()
    if AMP:
        torch.cuda.synchronize()
    return (time.time() - t0) / reps

k_all, _ = load_npz("data", TEST[0])
ks_s, ref_s = norm_slice(k_all[len(k_all) // 2])
m_s = make_mask(ks_s.shape[1], 8, "vd", SEEDS[0])
y_s = undersample(ks_s, m_s)
k_gpu, m_gpu = torch.from_numpy(ks_s).to(DEVICE)[None], torch.from_numpy(m_s).to(DEVICE)[None]
net8_cpu = UNet(CHANS, POOLS)
net8_cpu.load_state_dict({k: v.cpu() for k, v in MODELS["unet_R8"].state_dict().items()})
net8_cpu.eval()
k_cpu, m_cpu = torch.from_numpy(ks_s)[None], torch.from_numpy(m_s)[None]
lam_pc, it_pc = frozen_setting("per-coil (R=8)")
lam_mc, it_mc = frozen_setting("multi-coil (R=8)")

speed = [{"method": "zero-filled (CPU)", "seconds per slice": timed(lambda: recon_zf(y_s), 5)},
         {"method": f"U-Net ({'GPU' if AMP else 'CPU'})", "seconds per slice": timed(lambda: predict(MODELS["unet_R8"], k_gpu, m_gpu), 10)}]
if AMP:
    with torch.no_grad():
        speed.append({"method": "U-Net (CPU)", "seconds per slice": timed(lambda: net8_cpu(zero_filled_t(k_cpu, m_cpu)), 3)})
speed.append({"method": f"per-coil FISTA (CPU, {it_pc} iterations)", "seconds per slice": timed(lambda: recon_pc(y_s, m_s, lam_pc, it_pc), 1)})
t_maps = timed(lambda: espirit_maps(ks_s, **ESPIRIT), 1)
maps_s = espirit_maps(ks_s, **ESPIRIT)
t_mc = timed(lambda: recon_mc(y_s, m_s, maps_s, lam_mc, it_mc), 1)
speed.append({"method": f"multi-coil FISTA (CPU, {it_mc} iterations + ESPIRiT maps)", "seconds per slice": t_maps + t_mc})
out_s = predict(MODELS["unet_R8"], k_gpu, m_gpu).cpu().numpy()[0, 0]
t_dc = timed(lambda: data_consistency(out_s, y_s, m_s, maps_s), 3)
speed.append({"method": "U-Net + data consistency (U-Net on GPU, consistency step + ESPIRiT maps on CPU)",
              "seconds per slice": speed[1]["seconds per slice"] + t_maps + t_dc})
tableE = pd.DataFrame(speed)
tableE["milliseconds per slice"] = tableE["seconds per slice"] * 1000
tableE.to_csv(os.path.join(RES_DIR, "nb2_tableE_speed.csv"), index=False)
display(tableE.round(4))

fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.barh(tableE["method"], tableE["milliseconds per slice"], color="C2")
ax.bar_label(bars, fmt="%.0f ms", fontsize=8)
ax.set_xscale("log"); ax.invert_yaxis()
ax.set_xlabel("milliseconds per slice (log scale)"); ax.set_title("Reconstruction time per slice, R = 8")
plt.tight_layout()
save(fig, "nb2_fig04b_speed.png")

### 5e · Visual comparison
Held-out test slices at R = 8 and R = 4, all methods on the same mask. Error maps show where each method fails: errors along edges mean lost detail; structured errors that look like anatomy would mean the network invented or moved structure.

In [ ]:
def compare_figure(R, subj, pos, fname):
    k_all, slices = load_npz("data", subj)
    ks, ref = norm_slice(k_all[pos])
    fg = foreground_mask(ref)
    m = make_mask(ks.shape[1], R, "vd", SEEDS[0])
    y = undersample(ks, m)
    lam_pc, it_pc = frozen_setting(f"per-coil (R={R})")
    lam_mc, it_mc = frozen_setting(f"multi-coil (R={R})")
    maps = espirit_maps(ks, **ESPIRIT)
    unet_img = predict(MODELS[f"unet_R{R}"], torch.from_numpy(ks).to(DEVICE)[None],
                       torch.from_numpy(m).to(DEVICE)[None]).cpu().numpy()[0, 0]
    imgs = {"zero-filled": recon_zf(y),
            "per-coil FISTA": recon_pc(y, m, lam_pc, it_pc),
            "multi-coil FISTA": recon_mc(y, m, maps, lam_mc, it_mc),
            "U-Net": unet_img,
            "U-Net + data consistency": data_consistency(unet_img, y, m, maps)}
    fig, ax = plt.subplots(2, 6, figsize=(25, 9))
    show(ax[0, 0], ref, f"ground truth (subject {subj}, slice {slices[pos]})")
    ax[1, 0].axis("off")
    for j, (name, img) in enumerate(imgs.items(), start=1):
        s = metrics(img, ref, fg)
        show(ax[0, j], img, f"{name}, R = {R}\nPSNR {s['psnr']:.1f} dB  SSIM(knee) {s['ssim_fg']:.3f}")
        show(ax[1, j], np.abs(img - ref), "absolute error", vmax=0.3, cmap="inferno")
    plt.tight_layout()
    save(fig, fname)

subj_v = TEST[0]
pos_v = len(load_npz("data", subj_v)[1]) // 2
compare_figure(8, subj_v, pos_v, "nb2_fig04_visual_R8.png")
compare_figure(4, subj_v, pos_v, "nb2_fig05_visual_R4.png")

## Phase 6 · Statistical tests
Are the differences larger than chance would explain?

1. **One value per test slice:** each slice was reconstructed with several random masks; those scores are averaged first, so no slice is counted several times.
2. **Paired differences:** method A minus method B on the same slice.
3. **Wilcoxon signed-rank test** (two-sided) on the per-slice differences; it does not assume normally distributed differences.
4. **95 % bootstrap confidence interval** of the mean difference (10 000 resamples of slices).
5. **Holm correction** across all comparisons in the table, because many tests are run at once.
6. **Subject-level check:** slices of one person are not independent, so the number of test subjects whose mean difference has the same sign is reported with an exact sign test. With four subjects the smallest possible two-sided p-value is 0.125, so this check shows consistency but cannot reach significance on its own.

The comparisons below were fixed before looking at the results.

In [ ]:
from scipy import stats

PAIRS = [("U-Net", "multi-coil"), ("U-Net", "per-coil"), ("U-Net + data consistency", "U-Net"),
         ("multi-coil", "zero-filled"), ("per-coil", "zero-filled"), ("multi-coil", "per-coil")]
MLABEL = {"psnr": "PSNR (dB)", "ssim_fg": "SSIM inside knee"}

def holm(p):
    p = np.asarray(p, float)
    adj, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

def compare(df, a, b, metric, rng):
    wide = df.pivot_table(index=["subject", "pos"], columns="method", values=metric)
    d = (wide[a] - wide[b]).dropna()
    boots = d.values[rng.integers(0, len(d), (10_000, len(d)))].mean(axis=1)
    by_subject = d.groupby(level="subject").mean()
    k = int((by_subject > 0).sum())
    return {"slices": len(d), "mean Δ": d.mean(), "95% CI low": np.percentile(boots, 2.5),
            "95% CI high": np.percentile(boots, 97.5), "slices with Δ > 0": f"{(d > 0).sum()}/{len(d)}",
            "Wilcoxon p": stats.wilcoxon(d).pvalue, "subjects with Δ > 0": f"{k}/{len(by_subject)}",
            "subject sign-test p": stats.binomtest(k, len(by_subject), 0.5).pvalue}

per_slice = ALL[ALL.R.isin(R_TRAIN)].groupby(["subject", "pos", "R", "method"])[["psnr", "ssim_fg"]].mean().reset_index()
rng = np.random.default_rng(0)
rows = []
for R in R_TRAIN:
    dR = per_slice[per_slice.R == R]
    for metric in ["psnr", "ssim_fg"]:
        for a, b in PAIRS:
            rows.append({"R": R, "metric": MLABEL[metric], "comparison": f"{a} − {b}", **compare(dR, a, b, metric, rng)})
tableF = pd.DataFrame(rows)
tableF.insert(tableF.columns.get_loc("Wilcoxon p") + 1, "Holm-adjusted p", holm(tableF["Wilcoxon p"]))
tableF["significant (adjusted p < 0.05)"] = np.where(tableF["Holm-adjusted p"] < 0.05, "yes", "no")
tableF.to_csv(os.path.join(RES_DIR, "nb2_tableF_statistics.csv"), index=False)
display(tableF)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for j, metric in enumerate(MLABEL.values()):
    sub = tableF[tableF.metric == metric].reset_index(drop=True)
    y = np.arange(len(sub))[::-1]
    for yi, (_, r) in zip(y, sub.iterrows()):
        ax[j].errorbar(r["mean Δ"], yi, xerr=[[r["mean Δ"] - r["95% CI low"]], [r["95% CI high"] - r["mean Δ"]]],
                       fmt="o", color="C0" if r.R == R_TRAIN[0] else "C3", capsize=3)
    ax[j].axvline(0, color="gray", lw=0.8)
    ax[j].set_yticks(y)
    ax[j].set_yticklabels([f"R={r}: {cmp}" for r, cmp in zip(sub.R, sub.comparison)])
    ax[j].set_xlabel(f"mean difference in {metric} (95% CI)")
fig.suptitle(f"Paired differences on the test slices (blue: R = {R_TRAIN[0]}, red: R = {R_TRAIN[-1]})")
plt.tight_layout()
save(fig, "nb2_fig06_statistics.png")

coil = pd.DataFrame(json.load(open(os.path.join(NB1, "results", "coil_count.json"))))
cp = coil.pivot_table(index=["subject", "pos", "coils", "subset"], columns="method", values="psnr")
gain = (cp["multi-coil"] - cp["zero-filled"]).groupby(level=["subject", "pos", "coils"]).mean().unstack("coils")
lo_c, hi_c = gain.columns.min(), gain.columns.max()
dg = (gain[hi_c] - gain[lo_c]).dropna()
boots = dg.values[rng.integers(0, len(dg), (10_000, len(dg)))].mean(axis=1)
coil_test = pd.Series({"slices": len(dg), f"gain with {lo_c} coil(s), dB": gain[lo_c].mean(),
                       f"gain with {hi_c} coils, dB": gain[hi_c].mean(), "mean Δ (dB)": dg.mean(),
                       "95% CI low": np.percentile(boots, 2.5), "95% CI high": np.percentile(boots, 97.5),
                       "slices with Δ > 0": f"{(dg > 0).sum()}/{len(dg)}", "Wilcoxon p": stats.wilcoxon(dg).pvalue})
coil_test.to_frame("value").to_csv(os.path.join(RES_DIR, "nb2_tableG_coil_test.csv"))
print(f"Coil count (Notebook 1, R = {CFG['r_main'][-1]}): multi-coil gain with {hi_c} coils vs {lo_c} coil(s)")
display(coil_test.to_frame("value"))

**Reading the tests:** an interval that does not cross zero together with a small Holm-adjusted p-value means the difference is consistent across test slices and unlikely to be chance. The subject columns show whether all test people agree; with four subjects that check is descriptive. The p-values treat slices as the unit, so they are somewhat optimistic given that slices of one person are correlated.

## Phase 7 · Summary

In [ ]:
print("Main comparison (test subjects, variable-density masks):")
for R in R_TRAIN:
    dR = ALL[ALL.R == R]
    means = dR.groupby("method")[["psnr", "ssim_fg"]].mean()
    line = " | ".join(f"{m} {means.loc[m, 'psnr']:.2f} dB / {means.loc[m, 'ssim_fg']:.3f}"
                      for m in ["zero-filled", "per-coil", "multi-coil", "U-Net", "U-Net + data consistency"])
    print(f"  R = {R}: {line}")
    for m in ["U-Net", "U-Net + data consistency"]:
        d = paired(dR, m, "multi-coil")
        print(f"          {m} vs multi-coil: Δ PSNR {d.psnr.mean():+.2f} dB (better on {(d.psnr > 0).sum()}/{len(d)}), "
              f"Δ SSIM(knee) {d.ssim_fg.mean():+.4f} (better on {(d.ssim_fg > 0).sum()}/{len(d)})")
print("\nRobustness (mean PSNR, dB):")
print(tableC.round(2).to_string())
print("\nTraining data size at R = 8:")
print(tableD[["training subjects", "test PSNR", "test SSIM (knee)"]].round(3).to_string(index=False))
print("\nStatistical tests (Holm-adjusted):")
for _, r in tableF.iterrows():
    print(f"  R={r.R}  {r.metric:17s} {r.comparison:34s} Δ = {r['mean Δ']:+.4f} [{r['95% CI low']:+.4f}, {r['95% CI high']:+.4f}]"
          f"  adj. p = {r['Holm-adjusted p']:.2g}  ({r['subjects with Δ > 0']} subjects positive)")
print(f"  coil count: gain with {hi_c} vs {lo_c} coil(s): Δ = {coil_test['mean Δ (dB)']:+.2f} dB, Wilcoxon p = {coil_test['Wilcoxon p']:.2g}")
print("\nSpeed (ms per slice):")
print(tableE[["method", "milliseconds per slice"]].round(1).to_string(index=False))
print("\nFigures:", ", ".join(sorted(os.listdir(FIG_DIR))))
print("Results:", ", ".join(sorted(os.listdir(RES_DIR))))
print("Models:", ", ".join(sorted(f for f in os.listdir(MODEL_DIR) if f.endswith(".pt"))))

### Scope and assumptions
* The U-Net is trained on 14 subjects (40 slices each) from one scanner and protocol; it is evaluated on four other subjects from the same source. Generalisation to other scanners, contrasts or pathology is not tested.
* The U-Net works on magnitude images. Its plain output does not enforce agreement with the measured k-space lines; the data-consistency variant does, using a phase estimated from the fully sampled centre and the ESPIRiT coil maps.
* The statistical tests use the test slice as the unit; with four test subjects, subject-level conclusions are descriptive.
* Masks, normalisation and metrics are identical to Notebook 1 (verified by the zero-filled check in Phase 4). Normalisation uses the scale of the fully sampled image, as in Notebook 1, for all methods alike.
* FISTA settings come from Notebook 1 (tuned on the development subject); the U-Net checkpoint is chosen on the same subject.

### References
* Ronneberger O, Fischer P, Brox T. *U-Net: Convolutional networks for biomedical image segmentation.* MICCAI, 2015.
* Zbontar J et al. *fastMRI: An open dataset and benchmarks for accelerated MRI.* arXiv:1811.08839, 2018.
* Kingma DP, Ba J. *Adam: A method for stochastic optimization.* ICLR, 2015.
* Classical methods and data: see Notebook 1.